# Fama-French Regressions by Year and Decile
Este notebook recalcula as regressões de Fama-French de 3 Fatores para as carteiras (decis) formadas pelas métricas de rede (HRM e Pozzi).
A lógica atual utiliza os ativos agrupados em 10 decis para cada ano, construídos _In-Sample_ (ano $t$), e avaliados _Out-of-Sample_ (ano $t+1$).\n

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

## 1. Processamento e Regressões OLS
Aqui nós iteramos pelos anos (2014 a 2024), carregamos as rentabilidades diárias do ano seguinte, filtramos os tickers de cada decil e rodamos a regressão OLS.
Usamos o cálculo de **Equal-Weighted** log returns para os portfólios, garantindo que a carteira capte o efeito puramente estrutural da rede (sem enviesar por Market Cap).\n

In [15]:
# Carrega os fatores de Fama-French
factors_df = pd.read_parquet("../../data/02_clean/fama_french_factors.parquet")

# Os fatores originais geralmente vêm em porcentagem (ex: 1.5%), então dividimos por 100
factors_df = factors_df / 100

years = range(2014, 2025)
metrics = ['hrm', 'pozzi']
deciles = [f'decil_{i}' for i in range(1, 11)]

# Dicionário para armazenar resultados puros das regressões
regression_results = []

for metric in metrics:
    print(f"Processando regressões para: {metric.upper()}...")
    
    # Carrega metadados que dizem qual Ticker está em qual decil a cada ano
    try:
        df_meta = pd.read_parquet(f"../../data/07_portfolios_metadata/complete_metadata_{metric}.parquet")
        df_meta['year'] = df_meta['year'].astype(int)
    except FileNotFoundError:
        print(f"Arquivo de metadados para {metric} não encontrado. Pule.")
        continue
    
    for year in years:
        # Puxa retornos Out-of-Sample (ano + 1)
        try:
            oos_ret = pd.read_parquet(f"../../data/02_clean/returns_new_{year+1}.parquet")
        except FileNotFoundError:
            continue
            
        # Alinha os fatores de Fama-French às datas dos retornos daquele ano
        factors_year = factors_df[(factors_df.index >= oos_ret.index[0]) & (factors_df.index <= oos_ret.index[-1])]
        
        for decil in deciles:
            # Puxa tickers daquele decil no ano 'year'
            tickers = df_meta[(df_meta['year'] == year) & (df_meta['portfolio'] == decil)]['Ticker'].tolist()
            
            # Garante que as ações existam na base de retornos do ano t+1
            valid_tickers = [t for t in tickers if t in oos_ret.columns]
            
            if len(valid_tickers) == 0:
                continue
                
            # Calcula retorno Equal-Weighted da carteira
            # Primeiro tiramos log_returns e depois a média diária dos ativos
            port_ret = np.log1p(oos_ret[valid_tickers]).mean(axis=1)
            
            # Subtrai a Risk-Free rate para obter o Excesso de Retorno
            excess_ret = port_ret - factors_year['RF']
            
            # Variáveis independentes
            X = factors_year[['Mkt-RF', 'SMB', 'HML']]
            X = sm.add_constant(X)
            
            # Alinhamento por data (inner join) e drop de NAs
            aligned = pd.concat([excess_ret.rename("ExRet"), X], axis=1, join="inner").dropna()
            
            if len(aligned) < 30: # Evita regressões com poucos dias
                continue
                
            # Roda a Regressão OLS Clássica
            model = sm.OLS(aligned['ExRet'], aligned[['const', 'Mkt-RF', 'SMB', 'HML']]).fit()
            
            regression_results.append({
                'metric': metric,
                'year_oos': year + 1,      # Ano OOS (o que os retornos ocorreram)
                'year_formed': year,       # Ano em que a carteira foi formada
                'decil': decil,
                'alpha': model.params['const'] * 252, # Alpha Anualizado
                'alpha_tstat': model.tvalues['const'],
                'mkt_beta': model.params['Mkt-RF'],
                'smb_beta': model.params['SMB'],
                'hml_beta': model.params['HML'],
                'r_squared': model.rsquared
            })

df_results = pd.DataFrame(regression_results)
print("Todas as regressões foram calculadas!")

Processando regressões para: HRM...
Processando regressões para: POZZI...
Todas as regressões foram calculadas!


In [6]:
metric = "hrm"
pd.read_parquet(f"../../data/07_portfolios_metadata/complete_metadata_{metric}.parquet")

FileNotFoundError: [Errno 2] No such file or directory: '../../data/07_portfolios_metadata/complete_metadata_hrm.parquet'

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Carrega os fatores de Fama-French
factors_df = pd.read_parquet("../../data/02_clean/fama_french_factors.parquet")

# Os fatores originais geralmente vêm em porcentagem (ex: 1.5%), então dividimos por 100
factors_df = factors_df / 100

years = range(2014, 2025)
# Alterado para 'hcm', mas se o seu arquivo ainda for 'hrm', basta mudar aqui!
metrics = ['hcm', 'pozzi'] 
deciles = [f'decil_{i}' for i in range(1, 11)]

# Dicionário para armazenar resultados puros das regressões
regression_results = []

for metric in metrics:
    print(f"Processando regressões para: {metric.upper()}...")
    
    # Carrega metadados que dizem qual Ticker está em qual decil a cada ano
    try:
        df_meta = pd.read_parquet(f"../../data/07_portfolios_metadata/complete_metadata_{metric}.parquet")
        df_meta['year'] = df_meta['year'].astype(int)
    except FileNotFoundError:
        print(f"Arquivo de metadados para {metric} não encontrado. Pule.")
        continue
    
    for year in years:
        # Puxa retornos Out-of-Sample (ano + 1)
        try:
            oos_ret = pd.read_parquet(f"../../data/02_clean/returns_new_{year+1}.parquet")
        except FileNotFoundError:
            continue
            
        # Alinha os fatores de Fama-French às datas dos retornos daquele ano
        factors_year = factors_df[(factors_df.index >= oos_ret.index[0]) & (factors_df.index <= oos_ret.index[-1])]
        
        # Dicionário temporário para guardar os retornos das pontas do PMC
        port_returns_year = {}
        
        # =========================================================
        # 1. Regressões individuais para cada decil
        # =========================================================
        for decil in deciles:
            # Puxa tickers daquele decil no ano 'year'
            tickers = df_meta[(df_meta['year'] == year) & (df_meta['portfolio'] == decil)]['Ticker'].tolist()
            
            # Garante que as ações existam na base de retornos do ano t+1
            valid_tickers = [t for t in tickers if t in oos_ret.columns]
            
            if len(valid_tickers) == 0:
                continue
                
            # Calcula retorno Equal-Weighted da carteira
            port_ret = np.log1p(oos_ret[valid_tickers]).mean(axis=1)
            
            # Salva para criar o PMC depois
            if decil in ['decil_1', 'decil_10']:
                port_returns_year[decil] = port_ret
            
            # Subtrai a Risk-Free rate para obter o Excesso de Retorno
            excess_ret = port_ret - factors_year['RF']
            
            # Variáveis independentes
            X = factors_year[['Mkt-RF', 'SMB', 'HML']]
            X = sm.add_constant(X)
            
            # Alinhamento por data e drop de NAs
            aligned = pd.concat([excess_ret.rename("ExRet"), X], axis=1, join="inner").dropna()
            
            if len(aligned) < 30:
                continue
                
            # Roda a Regressão
            model = sm.OLS(aligned['ExRet'], aligned[['const', 'Mkt-RF', 'SMB', 'HML']]).fit()
            
            regression_results.append({
                'metric': metric,
                'year_oos': year + 1,
                'year_formed': year,
                'decil': decil,
                'alpha': model.params['const'] * 252, # Alpha Anualizado
                'alpha_tstat': model.tvalues['const'],
                'mkt_beta': model.params['Mkt-RF'],
                'smb_beta': model.params['SMB'],
                'hml_beta': model.params['HML'],
                'r_squared': model.rsquared
            })
            
        # =========================================================
        # 2. Regressão do Portfólio PMC (Peripheral Minus Central)
        # =========================================================
        if 'decil_10' in port_returns_year and 'decil_1' in port_returns_year:
            # Retorno do PMC = Peripheral(decil_10) - Central(decil_1)
            pmc_ret = port_returns_year['decil_10'] - port_returns_year['decil_1']
            
            # NOTA: O fator RF se anula em carteiras Long-Short, 
            # portanto o retorno líquido pmc_ret JÁ É O EXCESSO DE RETORNO!
            
            X = factors_year[['Mkt-RF', 'SMB', 'HML']]
            X = sm.add_constant(X)
            
            aligned_pmc = pd.concat([pmc_ret.rename("ExRet"), X], axis=1, join="inner").dropna()
            
            if len(aligned_pmc) >= 30:
                model_pmc = sm.OLS(aligned_pmc['ExRet'], aligned_pmc[['const', 'Mkt-RF', 'SMB', 'HML']]).fit()
                
                regression_results.append({
                    'metric': metric,
                    'year_oos': year + 1,
                    'year_formed': year,
                    'decil': 'PMC (10 - 1)', # Identificador único para essa carteira
                    'alpha': model_pmc.params['const'] * 252,
                    'alpha_tstat': model_pmc.tvalues['const'],
                    'mkt_beta': model_pmc.params['Mkt-RF'],
                    'smb_beta': model_pmc.params['SMB'],
                    'hml_beta': model_pmc.params['HML'],
                    'r_squared': model_pmc.rsquared
                })

df_results = pd.DataFrame(regression_results)
print("Todas as regressões foram calculadas, incluindo a carteira PMC!")

Processando regressões para: HCM...
Processando regressões para: POZZI...
Todas as regressões foram calculadas, incluindo a carteira PMC!


## 2. Opção de Tabela: Resumo Estilo Fama-MacBeth
Esta tabela sintetiza 10 anos de regressões. Ela exibe a **média temporal** dos Alphas e dos Betas para cada decil. 
O teste t (Alpha_tstat_FM) é calculado dividindo a média do Alpha pelo Erro Padrão da média (Desvio Padrão do Alpha / raiz do número de anos). Essa é a forma mais clássica de relatar performance em asset pricing.\n

In [3]:
def generate_fama_macbeth_table(df_metric_results, metric_name):
    if df_metric_results.empty:
        return None
        
    # Número de anos na amostra OOS
    n_years = df_metric_results['year_oos'].nunique()
    
    # Agrega tirando a média das variáveis e o desvio padrão do alpha
    # Como já incluímos o "PMC (10 - 1)" nas regressões originais, 
    # ele já entra neste cálculo automático junto com os decis!
    summary = df_metric_results.groupby('decil').agg({
        'alpha': ['mean', 'std'],
        'mkt_beta': 'mean',
        'smb_beta': 'mean',
        'hml_beta': 'mean',
        'r_squared': 'mean'
    })
    
    # Calcula T-Statistic no estilo Fama-MacBeth (Mean / Standard Error)
    summary['Alpha_tstat_FM'] = summary[('alpha', 'mean')] / (summary[('alpha', 'std')] / np.sqrt(n_years))
    
    # Achata as colunas (Flatten)
    summary.columns = ['Alpha', 'Alpha_Std', 'Mkt_Beta', 'SMB_Beta', 'HML_Beta', 'R_Squared', 'Alpha_tstat_FM']
    
    # Reordena para ficar bonito (decil_1 até decil_10 e depois a carteira Long-Short PMC)
    decil_order = [f'decil_{i}' for i in range(1, 11)]
    if 'PMC (10 - 1)' in summary.index:
        decil_order.append('PMC (10 - 1)')
    
    # Filtra e aplica a ordem correta
    summary = summary.reindex(decil_order)
    
    # Formata para visualização
    final_table = summary[['Alpha', 'Alpha_tstat_FM', 'Mkt_Beta', 'SMB_Beta', 'HML_Beta', 'R_Squared']].copy()
    final_table = final_table.round(4)
    
    # Renomeia o índice linha por linha com base no nome original
    index_names = []
    for idx in final_table.index:
        if idx == 'decil_1':
            index_names.append('Decil 1 (Central)')
        elif idx == 'decil_10':
            index_names.append('Decil 10 (Peripheral)')
        elif idx == 'PMC (10 - 1)':
            index_names.append('Long-Short (Peripheral - Central)')
        else:
            index_names.append(idx.replace('_', ' ').title())
            
    final_table.index = index_names
    
    print(f"\n== FAMA-MACBETH REGRESSION SUMMARY: {metric_name.upper()} ===")
    display(final_table)
    
    return final_table

# Executa para HCM e Pozzi (lembrando que atualizamos a nomenclatura!)
fm_hcm = generate_fama_macbeth_table(df_results[df_results['metric'] == 'hcm'], 'hcm')
fm_pozzi = generate_fama_macbeth_table(df_results[df_results['metric'] == 'pozzi'], 'pozzi')

# Salva tabelas
if fm_hcm is not None:
    fm_hcm.to_csv("../../data/07_portfolios_metadata/table_famamacbeth_hcm.csv")
if fm_pozzi is not None:
    fm_pozzi.to_csv("../../data/07_portfolios_metadata/table_famamacbeth_pozzi.csv")


== FAMA-MACBETH REGRESSION SUMMARY: HCM ===


,Alpha,Alpha_tstat_FM,Mkt_Beta,SMB_Beta,HML_Beta,R_Squared
Decil 1 (Central),-0.0812,-4.5970,0.9045,0.5874,0.3850,0.9642
Decil 2,-0.0552,-3.1513,0.9338,0.5611,0.3263,0.9687
Decil 3,-0.0638,-3.7579,0.8898,0.5285,0.3464,0.9681
Decil 4,-0.0787,-4.0762,0.8725,0.4825,0.3123,0.9643
Decil 5,-0.0701,-4.5440,0.8149,0.4431,0.2495,0.9459
Decil 6,-0.0780,-6.1512,0.8295,0.3944,0.2040,0.9246
Decil 7,-0.0743,-6.0923,0.7764,0.3591,0.0902,0.8860
Decil 8,-0.1135,-5.0865,0.7053,0.3933,0.0748,0.8307
Decil 9,-0.1368,-4.6163,0.5483,0.4498,0.0489,0.7433
Decil 10 (Peripheral),-0.0732,-2.0445,0.2838,0.2355,-0.0054,0.4493



== FAMA-MACBETH REGRESSION SUMMARY: POZZI ===


,Alpha,Alpha_tstat_FM,Mkt_Beta,SMB_Beta,HML_Beta,R_Squared
Decil 1 (Central),-0.0586,-3.0250,0.7817,0.2684,0.3205,0.8976
Decil 2,-0.0517,-3.3181,0.7642,0.3031,0.2839,0.8846
Decil 3,-0.0817,-3.6583,0.7964,0.3846,0.2427,0.9248
Decil 4,-0.0502,-4.4163,0.8229,0.4215,0.2048,0.9000
Decil 5,-0.0810,-3.5720,0.8117,0.4726,0.1968,0.8979
Decil 6,-0.0872,-3.7655,0.7870,0.4978,0.1746,0.8990
Decil 7,-0.0912,-3.8750,0.7910,0.5837,0.1924,0.9027
Decil 8,-0.0954,-4.4352,0.7744,0.5411,0.2022,0.9067
Decil 9,-0.1049,-4.8571,0.6512,0.4327,0.1429,0.7901
Decil 10 (Peripheral),-0.1227,-3.1853,0.5778,0.5294,0.0712,0.7319
